### Datos climatológicos
Como base, tomamos el listado de estaciones metereológicas del departamento de Boyacá  
Por medio del acceso a dhime (IDEAM) se descargan los datos disponibles en el departamento de Boyacá


In [1]:
# Manipulación de datos
import requests
import pandas as pd

# Visualización geoespacial
import folium
import geopandas as gpd
from shapely.geometry import Point

In [2]:
# Lectura de datos (Estaciones climatológicas en Colombia)
url = "https://www.datos.gov.co/resource/hp9r-jxuu.json"

params = {"$limit": 10000}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()
estaciones = pd.DataFrame(data)

estaciones.columns
# estaciones.head()
# estaciones.info()

Index(['codigo', 'nombre', 'categoria', 'tecnologia', 'estado', 'departamento',
       'municipio', 'ubicaci_n', 'altitud', 'longitud', 'latitud',
       'fecha_instalacion', 'area_operativa', 'area_hidrografica',
       'zona_hidrografica', 'subzona_hidrografica', 'entidad', 'corriente',
       'fecha_suspension'],
      dtype='object')

In [3]:
# Dataframe de estaciones
df_estaciones = estaciones.copy()

# Limpieza y tipificación de datos
df_estaciones.columns = (
    df_estaciones.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

columnas_numericas = ["codigo", "altitud", "longitud", "latitud"]

for col in columnas_numericas:
    if col in df_estaciones.columns:
        df_estaciones[col] = (
            df_estaciones[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.strip()
        )
        df_estaciones[col] = pd.to_numeric(df_estaciones[col], errors="coerce")

columnas_fecha = ["fecha_instalacion", "fecha_suspension"]

for col in columnas_fecha:
    if col in df_estaciones.columns:
        df_estaciones[col] = pd.to_datetime(
            df_estaciones[col],
            errors="coerce"
        )

columnas_texto = ["nombre", "categoria", "tecnologia", "estado", "departamento",  "municipio", "ubicaci_n", "area_operativa",
                  "area_hidrografica", "zona_hidrografica", "subzona_hidrografica", "entidad", "corriente"]

for col in columnas_texto:
    if col in df_estaciones.columns:
        df_estaciones[col] = (
            df_estaciones[col]
            .astype("string")
            .str.strip()
        )

columnas_categoricas = ["categoria", "tecnologia", "estado", "departamento", "municipio", "area_operativa",
                        "area_hidrografica", "zona_hidrografica", "subzona_hidrografica", "entidad"]

for col in columnas_categoricas:
    if col in df_estaciones.columns:
        df_estaciones[col] = df_estaciones[col].astype("category")

# Eliminación de duplicados
df_estaciones = df_estaciones.drop_duplicates(subset=["codigo"]).copy()

# Validación
df_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9464 entries, 0 to 9463
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   codigo                9464 non-null   int64         
 1   nombre                9464 non-null   string        
 2   categoria             9464 non-null   category      
 3   tecnologia            9464 non-null   category      
 4   estado                9464 non-null   category      
 5   departamento          9464 non-null   category      
 6   municipio             9464 non-null   category      
 7   ubicaci_n             9464 non-null   string        
 8   altitud               9464 non-null   int64         
 9   longitud              9464 non-null   float64       
 10  latitud               9464 non-null   float64       
 11  fecha_instalacion     9331 non-null   datetime64[ns]
 12  area_operativa        9464 non-null   category      
 13  area_hidrografica 

/tmp/ipykernel_32354/2764174296.py:28: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_estaciones[col] = pd.to_datetime(
/tmp/ipykernel_32354/2764174296.py:28: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_estaciones[col] = pd.to_datetime(


In [4]:
# Dataframe geoespacial de estaciones
gdf_estaciones_colombia = gpd.GeoDataFrame(
    df_estaciones,
    geometry=gpd.points_from_xy(
        df_estaciones["longitud"],
        df_estaciones["latitud"]
    ),
    crs="EPSG:4326"
)

gdf_estaciones_colombia.head()

,codigo,nombre,categoria,tecnologia,estado,departamento,municipio,ubicaci_n,altitud,longitud,latitud,fecha_instalacion,area_operativa,area_hidrografica,zona_hidrografica,subzona_hidrografica,entidad,corriente,fecha_suspension,geometry
0,35060220,LA GLORIA [35060220],Pluviométrica,Convencional,Activa,Cundinamarca,Ubalá,"{'latitude': '-73.41977778', 'longitude': '4.8...",1845,-73.419778,4.815694,1964-09-15,Area Operativa 11 - Cundinamarca-Amazonas,Orinoco,Meta,Río Guavio,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...,<NA>,NaT,POINT (-73.41978 4.81569)
1,23197510,LADRILLERA [23197510],Limnimétrica,Convencional,Activa,Santander,Bucaramanga,"{'latitude': '-73.13333333', 'longitude': '7.1...",880,-73.133333,7.116667,1981-09-15,Area Operativa 08 - Santanderes-Arauca,Magdalena Cauca,Medio Magdalena,Río Lebrija y otros directos al Magdalena,PROYECTO COLOMBO - HOLANDÉS,QUEBRADA IGLESIA,NaT,POINT (-73.13333 7.11667)
2,2615700040,AGUAS DE MANIZALES - AUT [2615700040],Limnigráfica,Automática con Telemetría,Activa,Caldas,Manizales,"{'latitude': '-75.490305556', 'longitude': '5....",1659,-75.490306,5.089917,2018-07-04,Area Operativa 09 - Cauca-Valle-Caldas,Magdalena Cauca,Cauca,Río Chinchiná,CORPORACIÓN AUTÓNOMA REGIONAL DE CALDAS,QUEBRADA EL GUAMO,NaT,POINT (-75.49031 5.08992)
3,26090930,PICHICHI [26090930],Pluviométrica,Convencional,Activa,Valle Del Cauca,Guacarí,"{'latitude': '-76.28333333', 'longitude': '3.7...",1029,-76.283333,3.783333,1969-01-15,Area Operativa 09 - Cauca-Valle-Caldas,Magdalena Cauca,Cauca,"Ríos Guabas,Sabaletas y Sonso",ESTACIONES PARTICULARES,<NA>,NaT,POINT (-76.28333 3.78333)
4,3503500395,LAGUNA NEGRA AUT - [3503500395],Climatológica Principal,Automática con Telemetría,Activa,Cundinamarca,Fómeque,"{'latitude': '-73.771518889', 'longitude': '4....",3719,-73.771519,4.591235,2023-12-11,Area Operativa 11 - Cundinamarca-Amazonas,Orinoco,Meta,Río Guatiquía,PARQUES NACIONALES NATURALES,<NA>,NaT,POINT (-73.77152 4.59124)


In [5]:
departamento = "Boyacá"

gdf_estaciones_dpto = gdf_estaciones_colombia[gdf_estaciones_colombia["departamento"].astype(str).eq(departamento)].copy()

print(f"Total de estaciones seleccionadas en {departamento}: {len(gdf_estaciones_dpto)}")
gdf_estaciones_dpto["categoria"].value_counts()

Total de estaciones seleccionadas en Boyacá: 551


categoria
Pluviométrica              182
Limnimétrica               151
Climatológica Ordinaria     67
Climatológica Principal     49
Limnigráfica                45
Pluviográfica               32
Meteorológica Especial      20
Agrometeorológica            5
HidroMeteorologica           0
Ambiental                    0
Meteorologica Marina         0
Radio Sonda                  0
Sinóptica Principal          0
Sinóptica Secundaria         0
Name: count, dtype: int64

### Funcionalidad con base al tipo de estación 

* **Pluviométrica:** Mide precipitación acumulada (sin intensidad temporal detallada)  
* **Limnimétrica:** Mide nivel de agua en ríos o cuerpos hídricos (disponibilidad hídrica)
* **Climatológica ordinaria:** Mide temperatura, lluvia, humedad
* **Climatológica principal:** Mide precipitación, temperatura, humedad, viento, radiación solar, evaporación, presión atmosférica
* **Limnigráfica:** Mide nivel de agua continuo en el tiempo
* **Pluviográfica:** Mide precipitación, intensidad de lluvia en el tiempo, distribución temporal del viento (Extremos)
* **Metereológica especial:** Mide viento, radiación, evapotransporación
* **Agrometereológica:** Mide temperatura, humedad, precipitación, radiación, humedad del suelo, evapotransporación (Especial agro)

In [6]:
# Centro aproximado del departamento
centro_mapa = [
    gdf_estaciones_dpto["latitud"].mean(),
    gdf_estaciones_dpto["longitud"].mean()
]

# Crear mapa base
m = folium.Map(
    location=centro_mapa,
    zoom_start=8,
    tiles="CartoDB positron"
)

# Colores por categoría
colores_categoria = {
    "Pluviométrica": "blue",
    "Pluviográfica": "darkblue",
    "Climatológica Principal": "green",
    "Climatológica Ordinaria": "lightgreen",
    "Agrometeorológica": "orange",
    "Meteorológica Especial": "purple",
    "Limnimétrica": "red",
    "Limnigráfica": "darkred"
}

# Agregar estaciones
for _, row in gdf_estaciones_dpto.iterrows():
    
    categoria = row["categoria"]
    color = colores_categoria.get(categoria, "gray")

    popup_text = f"""
    <b>Nombre:</b> {row['nombre']}<br>
    <b>Código:</b> {row['codigo']}<br>
    <b>Categoría:</b> {row['categoria']}<br>
    <b>Municipio:</b> {row['municipio']}<br>
    <b>Altitud:</b> {row['altitud']} msnm<br>
    <b>Estado:</b> {row['estado']}
    """

    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Mostrar mapa
m